# 59: Risk-Adjusted Performance Metrics

## Executive Summary

This notebook implements **professional risk-adjusted performance analysis** as practiced at institutional asset managers (BlackRock, Vanguard) and hedge fund due diligence (Cambridge Associates).

### What Performance Analysts Need:

**1. Return Metrics**
- Total return (CAGR)
- Arithmetic vs geometric mean
- Excess return (alpha)
- Rolling returns

**2. Risk Metrics**
- Volatility (standard deviation)
- Downside deviation
- Maximum drawdown
- Value at Risk (VaR)

**3. Risk-Adjusted Ratios**
- Sharpe Ratio
- Sortino Ratio
- Calmar Ratio
- Information Ratio
- Treynor Ratio

**4. Benchmark Comparison**
- Alpha, Beta
- Tracking error
- Active share
- Up/Down capture

**Data sources**: `qj.eod.get_historical_prices`, local pandas performance analytics

**API:** https://api.quantjourney.cloud

## Run Output

![59_risk_adjusted_performance](../plots/59_risk_adjusted_performance_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# =============================================================================
# SETUP & CONFIGURATION
# =============================================================================

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

from quantjourney.sdk import QuantJourneyAPI
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# API Connection
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)

# Configuration
pd.set_option('display.float_format', '{:,.4f}'.format)
np.random.seed(42)

# Risk-free rate assumption
RISK_FREE_RATE = 0.045  # 4.5% annual

print("="*80)
print("RISK-ADJUSTED PERFORMANCE METRICS")
print("Professional Performance Analysis")
print("="*80)
print(f"\n✓ Connected to QuantJourney API")
print(f"✓ Risk-Free Rate: {RISK_FREE_RATE*100:.1f}%")
print(f"✓ Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


---

## Section 1: Performance Data Collection

### Analyst Perspective (Morningstar Research)

**Portfolio/Fund Selection:**
- Various strategies with different risk profiles
- Minimum 3-5 years of data
- Include appropriate benchmarks
- Adjusted for dividends/distributions

**Data Quality:**
- NAV-based returns preferred
- Account for survivorship bias
- Verify calculation methodology

In [ ]:
# =============================================================================
# SECTION 1: DATA COLLECTION
# =============================================================================

print("\n" + "="*80)
print("PERFORMANCE DATA COLLECTION")
print("="*80)

# Funds/ETFs representing different strategies
funds = {
    'SPY': 'S&P 500 ETF (Benchmark)',
    'QQQ': 'Nasdaq 100 ETF (Growth)',
    'VTV': 'Vanguard Value ETF',
    'MTUM': 'iShares Momentum Factor ETF',
    'QUAL': 'iShares Quality Factor ETF',
    'USMV': 'iShares Min Vol ETF',
    'TLT': 'Long Treasury Bond ETF'
}

print(f"\n[1.1] Fetching Performance Data")
print("-" * 50)

price_data = {}
data_loaded = False

end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=5*365)).strftime('%Y-%m-%d')

for symbol in funds.keys():
    try:
        response = qj.eod.get_historical_prices(
            symbol=symbol,
            start_date=start_date,
            end_date=end_date,
            frequency='1d'
        )
        data = response.get('data', response.get('value', response)) if isinstance(response, dict) else response
        if isinstance(data, dict):
            data = data.get(symbol) or data.get(symbol.upper()) or data.get('prices') or data.get('rows') or data.get('results') or data
        if isinstance(data, dict):
            data = [data]
        
        if isinstance(data, list) and len(data) > 500:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date').sort_index()
            price_col = 'adjusted_close' if 'adjusted_close' in df.columns else 'close'
            price_data[symbol] = pd.to_numeric(df[price_col], errors='coerce')
            data_loaded = True
            print(f"     ✓ {symbol}: {len(df)} days")
    except:
        pass

if not data_loaded or len(price_data) < 4:
    print("     Generating synthetic performance data...")
    
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    n = len(dates)
    
    # Performance characteristics (annual return, volatility, beta)
    characteristics = {
        'SPY': {'ret': 0.10, 'vol': 0.15, 'beta': 1.0},
        'QQQ': {'ret': 0.15, 'vol': 0.22, 'beta': 1.2},
        'VTV': {'ret': 0.08, 'vol': 0.14, 'beta': 0.9},
        'MTUM': {'ret': 0.12, 'vol': 0.18, 'beta': 1.05},
        'QUAL': {'ret': 0.11, 'vol': 0.16, 'beta': 0.95},
        'USMV': {'ret': 0.08, 'vol': 0.11, 'beta': 0.70},
        'TLT': {'ret': 0.02, 'vol': 0.15, 'beta': -0.2}
    }
    
    # Generate SPY first (benchmark)
    spy_daily_ret = characteristics['SPY']['ret'] / 252
    spy_daily_vol = characteristics['SPY']['vol'] / np.sqrt(252)
    spy_returns = np.random.normal(spy_daily_ret, spy_daily_vol, n)
    
    # Add some regime shifts
    for i in range(n):
        if i > n//3 and i < n//3 + 50:  # Market correction
            spy_returns[i] -= 0.003
    
    spy_prices = 400 * np.exp(np.cumsum(spy_returns))
    price_data['SPY'] = pd.Series(spy_prices, index=dates)
    
    # Generate other funds with correlation to SPY
    for symbol, chars in characteristics.items():
        if symbol == 'SPY':
            continue
            
        beta = chars['beta']
        alpha = (chars['ret'] - beta * characteristics['SPY']['ret']) / 252
        idio_vol = np.sqrt(chars['vol']**2 - (beta * characteristics['SPY']['vol'])**2) / np.sqrt(252)
        
        returns = alpha + beta * spy_returns + np.random.normal(0, max(idio_vol, 0.005), n)
        prices = 100 * np.exp(np.cumsum(returns))
        price_data[symbol] = pd.Series(prices, index=dates)
        
    for sym in price_data.keys():
        print(f"     ✓ {sym}: {len(price_data[sym])} days (synthetic)")


In [ ]:
# =============================================================================
# CALCULATE RETURNS
# =============================================================================

print("\n[1.2] Computing Returns")
print("-" * 50)

# Align all series
prices_df = pd.DataFrame(price_data)
prices_df = prices_df.dropna()

# Daily returns
returns_df = prices_df.pct_change().dropna()

# Monthly returns
monthly_returns = prices_df.resample('M').last().pct_change().dropna()

print(f"     Data Period: {returns_df.index[0].strftime('%Y-%m-%d')} to {returns_df.index[-1].strftime('%Y-%m-%d')}")
print(f"     Trading Days: {len(returns_df)}")
print(f"     Months: {len(monthly_returns)}")


---

## Section 2: Risk-Adjusted Ratios

### Analyst Perspective (Cambridge Associates)

**Key Ratios:**

1. **Sharpe Ratio:**
$$SR = \frac{R_p - R_f}{\sigma_p}$$

2. **Sortino Ratio:**
$$Sortino = \frac{R_p - R_f}{\sigma_d}$$
where $\sigma_d$ = downside deviation

3. **Calmar Ratio:**
$$Calmar = \frac{CAGR}{|MaxDrawdown|}$$

4. **Information Ratio:**
$$IR = \frac{R_p - R_b}{\sigma_{tracking}}$$

5. **Treynor Ratio:**
$$TR = \frac{R_p - R_f}{\beta_p}$$

In [ ]:
# =============================================================================
# SECTION 2: RISK-ADJUSTED METRICS
# =============================================================================

print("\n" + "="*80)
print("RISK-ADJUSTED PERFORMANCE METRICS")
print("="*80)

class PerformanceMetrics:
    """
    Professional performance analytics.
    """
    
    def __init__(self, returns, benchmark_returns=None, rf_rate=0.045):
        """
        Args:
            returns: Daily returns series
            benchmark_returns: Benchmark daily returns
            rf_rate: Annual risk-free rate
        """
        self.returns = returns
        self.benchmark = benchmark_returns
        self.rf_daily = rf_rate / 252
        self.rf_annual = rf_rate
        self.n_periods = len(returns)
    
    # ========== Return Metrics ==========
    
    def total_return(self):
        """Cumulative total return."""
        return (1 + self.returns).prod() - 1
    
    def cagr(self):
        """Compound Annual Growth Rate."""
        total = self.total_return()
        years = self.n_periods / 252
        return (1 + total) ** (1 / years) - 1
    
    def arithmetic_mean(self):
        """Annualized arithmetic mean."""
        return self.returns.mean() * 252
    
    # ========== Risk Metrics ==========
    
    def volatility(self):
        """Annualized volatility."""
        return self.returns.std() * np.sqrt(252)
    
    def downside_deviation(self, mar=None):
        """
        Downside deviation (semi-deviation).
        
        Args:
            mar: Minimum acceptable return (default: risk-free)
        """
        if mar is None:
            mar = self.rf_daily
        
        downside = np.minimum(self.returns - mar, 0)
        return np.sqrt((downside ** 2).mean()) * np.sqrt(252)
    
    def max_drawdown(self):
        """Maximum drawdown."""
        cumulative = (1 + self.returns).cumprod()
        rolling_max = cumulative.cummax()
        drawdown = cumulative / rolling_max - 1
        return drawdown.min()
    
    def var(self, confidence=0.95):
        """Value at Risk (historical)."""
        return np.percentile(self.returns, (1 - confidence) * 100)
    
    def cvar(self, confidence=0.95):
        """Conditional VaR (Expected Shortfall)."""
        var = self.var(confidence)
        return self.returns[self.returns <= var].mean()
    
    # ========== Risk-Adjusted Ratios ==========
    
    def sharpe_ratio(self):
        """Sharpe Ratio."""
        excess_return = self.cagr() - self.rf_annual
        return excess_return / self.volatility()
    
    def sortino_ratio(self):
        """Sortino Ratio."""
        excess_return = self.cagr() - self.rf_annual
        return excess_return / self.downside_deviation()
    
    def calmar_ratio(self):
        """Calmar Ratio."""
        return self.cagr() / abs(self.max_drawdown())
    
    def treynor_ratio(self):
        """Treynor Ratio."""
        if self.benchmark is None:
            return np.nan
        beta = self.beta()
        if beta == 0:
            return np.nan
        excess_return = self.cagr() - self.rf_annual
        return excess_return / beta
    
    def information_ratio(self):
        """Information Ratio."""
        if self.benchmark is None:
            return np.nan
        
        active_returns = self.returns - self.benchmark
        active_return_ann = active_returns.mean() * 252
        tracking_error = active_returns.std() * np.sqrt(252)
        
        return active_return_ann / tracking_error
    
    # ========== Benchmark Metrics ==========
    
    def beta(self):
        """Beta vs benchmark."""
        if self.benchmark is None:
            return np.nan
        
        covariance = np.cov(self.returns, self.benchmark)[0, 1]
        variance = self.benchmark.var()
        
        return covariance / variance
    
    def alpha(self):
        """Jensen's Alpha."""
        if self.benchmark is None:
            return np.nan
        
        # Alpha = Rp - [Rf + Beta * (Rm - Rf)]
        bm_return = (1 + self.benchmark).prod() ** (252 / len(self.benchmark)) - 1
        expected = self.rf_annual + self.beta() * (bm_return - self.rf_annual)
        
        return self.cagr() - expected
    
    def tracking_error(self):
        """Tracking error."""
        if self.benchmark is None:
            return np.nan
        
        active_returns = self.returns - self.benchmark
        return active_returns.std() * np.sqrt(252)
    
    def up_capture(self):
        """Up market capture ratio."""
        if self.benchmark is None:
            return np.nan
        
        up_periods = self.benchmark > 0
        if up_periods.sum() == 0:
            return np.nan
        
        fund_up = self.returns[up_periods].mean()
        bm_up = self.benchmark[up_periods].mean()
        
        return fund_up / bm_up
    
    def down_capture(self):
        """Down market capture ratio."""
        if self.benchmark is None:
            return np.nan
        
        down_periods = self.benchmark < 0
        if down_periods.sum() == 0:
            return np.nan
        
        fund_down = self.returns[down_periods].mean()
        bm_down = self.benchmark[down_periods].mean()
        
        return fund_down / bm_down
    
    def capture_ratio(self):
        """Up/Down capture ratio."""
        up = self.up_capture()
        down = self.down_capture()
        if down == 0 or np.isnan(down):
            return np.nan
        return up / down
    
    # ========== Summary ==========
    
    def summary(self):
        """Complete performance summary."""
        return {
            'Total Return': self.total_return(),
            'CAGR': self.cagr(),
            'Volatility': self.volatility(),
            'Sharpe Ratio': self.sharpe_ratio(),
            'Sortino Ratio': self.sortino_ratio(),
            'Calmar Ratio': self.calmar_ratio(),
            'Max Drawdown': self.max_drawdown(),
            'VaR (95%)': self.var(),
            'CVaR (95%)': self.cvar(),
            'Beta': self.beta(),
            'Alpha': self.alpha(),
            'Tracking Error': self.tracking_error(),
            'Information Ratio': self.information_ratio(),
            'Treynor Ratio': self.treynor_ratio(),
            'Up Capture': self.up_capture(),
            'Down Capture': self.down_capture()
        }

# Calculate metrics for all funds
print("\n[2.1] Risk-Adjusted Metrics")
print("-" * 50)

benchmark_returns = returns_df['SPY']

metrics_list = []
for symbol in returns_df.columns:
    pm = PerformanceMetrics(
        returns_df[symbol],
        benchmark_returns if symbol != 'SPY' else None,
        rf_rate=RISK_FREE_RATE
    )
    summary = pm.summary()
    summary['Symbol'] = symbol
    summary['Strategy'] = funds.get(symbol, symbol)
    metrics_list.append(summary)

metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df.set_index('Symbol')

# Display key metrics
display_cols = ['CAGR', 'Volatility', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown']
display_df = metrics_df[display_cols].copy()
display_df['CAGR'] = display_df['CAGR'] * 100
display_df['Volatility'] = display_df['Volatility'] * 100
display_df['Max Drawdown'] = display_df['Max Drawdown'] * 100

print(display_df.round(2).to_string())


In [ ]:
# =============================================================================
# BENCHMARK COMPARISON
# =============================================================================

print("\n[2.2] Benchmark Comparison (vs SPY)")
print("-" * 50)

bm_cols = ['Beta', 'Alpha', 'Tracking Error', 'Information Ratio', 'Up Capture', 'Down Capture']
bm_df = metrics_df[bm_cols].copy()
bm_df['Alpha'] = bm_df['Alpha'] * 100
bm_df['Tracking Error'] = bm_df['Tracking Error'] * 100

print(bm_df.round(3).to_string())


In [ ]:
# =============================================================================
# PERFORMANCE VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Cumulative Returns',
        'Risk-Return Tradeoff',
        'Sharpe Ratio Comparison',
        'Drawdown Analysis'
    ]
)

# 1. Cumulative returns
cum_returns = (1 + returns_df).cumprod()
for symbol in cum_returns.columns:
    fig.add_trace(go.Scatter(
        x=cum_returns.index,
        y=(cum_returns[symbol] - 1) * 100,
        name=symbol,
        mode='lines'
    ), row=1, col=1)

# 2. Risk-return scatter
fig.add_trace(go.Scatter(
    x=metrics_df['Volatility'] * 100,
    y=metrics_df['CAGR'] * 100,
    mode='markers+text',
    marker=dict(size=15, color='cyan'),
    text=metrics_df.index,
    textposition='top center',
    showlegend=False
), row=1, col=2)

# Capital market line
vol_range = np.linspace(0, 0.25, 50)
spy_sharpe = metrics_df.loc['SPY', 'Sharpe Ratio']
cml = RISK_FREE_RATE + spy_sharpe * vol_range
fig.add_trace(go.Scatter(
    x=vol_range * 100,
    y=cml * 100,
    mode='lines',
    line=dict(dash='dash', color='white'),
    name='CML',
    showlegend=False
), row=1, col=2)

# 3. Sharpe ratios
sorted_sharpe = metrics_df['Sharpe Ratio'].sort_values(ascending=True)
colors = ['lime' if s > 0.5 else 'yellow' if s > 0 else 'red' for s in sorted_sharpe]
fig.add_trace(go.Bar(
    x=sorted_sharpe.values,
    y=sorted_sharpe.index,
    orientation='h',
    marker_color=colors,
    showlegend=False
), row=2, col=1)

# 4. Drawdowns
for symbol in ['SPY', 'QQQ', 'USMV']:
    cumulative = (1 + returns_df[symbol]).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative / rolling_max - 1) * 100
    fig.add_trace(go.Scatter(
        x=drawdown.index,
        y=drawdown,
        name=f'{symbol} DD',
        fill='tozeroy'
    ), row=2, col=2)

fig.update_layout(
    title=dict(text='Risk-Adjusted Performance Analysis', font=dict(size=20)),
    template='plotly_dark',
    height=700
)

fig.update_yaxes(title_text='Cumulative Return (%)', row=1, col=1)
fig.update_xaxes(title_text='Volatility (%)', row=1, col=2)
fig.update_yaxes(title_text='CAGR (%)', row=1, col=2)
fig.update_xaxes(title_text='Sharpe Ratio', row=2, col=1)
fig.update_yaxes(title_text='Drawdown (%)', row=2, col=2)

fig.show()


---

## Section 3: Rolling Analysis

### Analyst Perspective (BlackRock Portfolio Analytics)

**Rolling Metrics:**
- Rolling returns (12-month, 36-month)
- Rolling volatility
- Rolling Sharpe ratio
- Rolling beta

**Why Rolling Analysis:**
- Assess consistency of performance
- Identify regime changes
- Evaluate manager skill vs luck

In [ ]:
# =============================================================================
# SECTION 3: ROLLING ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("ROLLING PERFORMANCE ANALYSIS")
print("="*80)

def rolling_metrics(returns, window=252):
    """Calculate rolling performance metrics."""
    rolling_return = returns.rolling(window).apply(lambda x: (1 + x).prod() - 1)
    rolling_vol = returns.rolling(window).std() * np.sqrt(252)
    rolling_sharpe = (rolling_return - RISK_FREE_RATE) / rolling_vol
    
    return rolling_return, rolling_vol, rolling_sharpe

print("\n[3.1] Rolling Metrics (12-Month Window)")
print("-" * 50)

# Calculate for main funds
rolling_data = {}
for symbol in ['SPY', 'QQQ', 'USMV']:
    ret, vol, sharpe = rolling_metrics(returns_df[symbol], window=252)
    rolling_data[symbol] = {
        'return': ret,
        'vol': vol,
        'sharpe': sharpe
    }

# Summary stats of rolling metrics
for symbol in rolling_data:
    sharpe = rolling_data[symbol]['sharpe'].dropna()
    print(f"     {symbol} Rolling Sharpe: Mean={sharpe.mean():.2f}, Min={sharpe.min():.2f}, Max={sharpe.max():.2f}")


In [ ]:
# =============================================================================
# ROLLING VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Rolling 12-Month Return',
        'Rolling 12-Month Volatility',
        'Rolling 12-Month Sharpe Ratio',
        'Rolling Beta (QQQ vs SPY)'
    ]
)

# 1. Rolling returns
for symbol in rolling_data:
    fig.add_trace(go.Scatter(
        x=rolling_data[symbol]['return'].index,
        y=rolling_data[symbol]['return'] * 100,
        name=symbol
    ), row=1, col=1)

# 2. Rolling volatility
for symbol in rolling_data:
    fig.add_trace(go.Scatter(
        x=rolling_data[symbol]['vol'].index,
        y=rolling_data[symbol]['vol'] * 100,
        name=symbol,
        showlegend=False
    ), row=1, col=2)

# 3. Rolling Sharpe
for symbol in rolling_data:
    fig.add_trace(go.Scatter(
        x=rolling_data[symbol]['sharpe'].index,
        y=rolling_data[symbol]['sharpe'],
        name=symbol,
        showlegend=False
    ), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='white', row=2, col=1)

# 4. Rolling beta
def rolling_beta(returns, benchmark, window=252):
    def calc_beta(ret, bm):
        cov = np.cov(ret, bm)[0, 1]
        var = bm.var()
        return cov / var if var > 0 else np.nan
    
    betas = []
    for i in range(window, len(returns)):
        beta = calc_beta(
            returns.iloc[i-window:i].values,
            benchmark.iloc[i-window:i].values
        )
        betas.append(beta)
    
    return pd.Series(betas, index=returns.index[window:])

qqq_beta = rolling_beta(returns_df['QQQ'], returns_df['SPY'])
fig.add_trace(go.Scatter(
    x=qqq_beta.index,
    y=qqq_beta.values,
    name='QQQ Beta',
    line=dict(color='cyan'),
    showlegend=False
), row=2, col=2)
fig.add_hline(y=1.0, line_dash='dash', line_color='white', row=2, col=2)

fig.update_layout(
    title=dict(text='Rolling Performance Analysis', font=dict(size=20)),
    template='plotly_dark',
    height=700
)

fig.update_yaxes(title_text='Return (%)', row=1, col=1)
fig.update_yaxes(title_text='Volatility (%)', row=1, col=2)
fig.update_yaxes(title_text='Sharpe Ratio', row=2, col=1)
fig.update_yaxes(title_text='Beta', row=2, col=2)

fig.show()


---

## Section 4: Performance Attribution

### Analyst Perspective (Vanguard Quantitative Research)

**Return Attribution:**

$$R_p = \alpha + \beta R_m + \epsilon$$

Or with multiple factors:

$$R_p = \alpha + \sum_i \beta_i F_i + \epsilon$$

**Skill vs Luck:**
- T-statistic of alpha
- Persistence of returns
- Hit rate analysis

In [ ]:
# =============================================================================
# SECTION 4: PERFORMANCE ATTRIBUTION
# =============================================================================

print("\n" + "="*80)
print("PERFORMANCE ATTRIBUTION")
print("="*80)

print("\n[4.1] Regression Analysis")
print("-" * 50)

def capm_regression(returns, benchmark, rf_daily):
    """CAPM regression: Rp - Rf = alpha + beta * (Rm - Rf)."""
    excess_ret = returns - rf_daily
    excess_bm = benchmark - rf_daily
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        excess_bm, excess_ret
    )
    
    return {
        'Alpha (Daily)': intercept,
        'Alpha (Annual)': intercept * 252,
        'Beta': slope,
        'R-squared': r_value ** 2,
        'Alpha t-stat': intercept / std_err if std_err > 0 else np.nan,
        'p-value': p_value
    }

rf_daily = RISK_FREE_RATE / 252

regression_results = []
for symbol in returns_df.columns:
    if symbol == 'SPY':
        continue
    
    result = capm_regression(returns_df[symbol], returns_df['SPY'], rf_daily)
    result['Symbol'] = symbol
    regression_results.append(result)

regression_df = pd.DataFrame(regression_results)
regression_df = regression_df.set_index('Symbol')

print(regression_df[['Alpha (Annual)', 'Beta', 'R-squared', 'Alpha t-stat']].round(3).to_string())


In [ ]:
# =============================================================================
# HIT RATE ANALYSIS
# =============================================================================

print("\n[4.2] Hit Rate Analysis")
print("-" * 50)

def hit_rate_analysis(returns, benchmark):
    """Analyze outperformance frequency."""
    # Daily
    daily_beat = (returns > benchmark).mean()
    
    # Monthly
    monthly_ret = returns.resample('M').apply(lambda x: (1 + x).prod() - 1)
    monthly_bm = benchmark.resample('M').apply(lambda x: (1 + x).prod() - 1)
    monthly_beat = (monthly_ret > monthly_bm).mean()
    
    # Yearly
    yearly_ret = returns.resample('Y').apply(lambda x: (1 + x).prod() - 1)
    yearly_bm = benchmark.resample('Y').apply(lambda x: (1 + x).prod() - 1)
    yearly_beat = (yearly_ret > yearly_bm).mean()
    
    return {
        'Daily Hit Rate': daily_beat,
        'Monthly Hit Rate': monthly_beat,
        'Yearly Hit Rate': yearly_beat
    }

hit_rates = []
for symbol in returns_df.columns:
    if symbol == 'SPY':
        continue
    
    result = hit_rate_analysis(returns_df[symbol], returns_df['SPY'])
    result['Symbol'] = symbol
    hit_rates.append(result)

hit_rate_df = pd.DataFrame(hit_rates)
hit_rate_df = hit_rate_df.set_index('Symbol')

print((hit_rate_df * 100).round(1).to_string())


---

## Section 5: Executive Summary

In [ ]:
# =============================================================================
# SECTION 5: EXECUTIVE SUMMARY
# =============================================================================

print("\n" + "="*80)
print("PERFORMANCE ANALYSIS EXECUTIVE SUMMARY")
print("="*80)

# Rank by Sharpe ratio
ranked = metrics_df.sort_values('Sharpe Ratio', ascending=False)

print(f"""
┌──────────────────────────────────────────────────────────────────────────────┐
│  RISK-ADJUSTED PERFORMANCE REPORT                                            │
│  Report Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}                                             │
│  Analysis Period: {returns_df.index[0].strftime('%Y-%m-%d')} to {returns_df.index[-1].strftime('%Y-%m-%d')}                          │
├──────────────────────────────────────────────────────────────────────────────┤
│  RISK-FREE RATE: {RISK_FREE_RATE*100:.1f}%                                                     │
├──────────────────────────────────────────────────────────────────────────────┤
│  PERFORMANCE RANKINGS (by Sharpe Ratio)                                      │
│                                                                              │
│    Rank  Symbol  CAGR     Vol      Sharpe   Max DD                           │
│    ────  ──────  ───────  ───────  ──────   ──────                           │
""")

for i, (symbol, row) in enumerate(ranked.head(7).iterrows()):
    print(f"│    {i+1:>2d}.   {symbol:6s}  {row['CAGR']*100:>6.1f}%  {row['Volatility']*100:>6.1f}%  {row['Sharpe Ratio']:>6.2f}   {row['Max Drawdown']*100:>6.1f}%                         │")

print(f"""
├──────────────────────────────────────────────────────────────────────────────┤
│  BENCHMARK COMPARISON (vs SPY)                                               │
│                                                                              │
│    TOP ALPHA GENERATORS:                                                     │""")

alpha_ranked = regression_df.sort_values('Alpha (Annual)', ascending=False)
for symbol, row in alpha_ranked.head(3).iterrows():
    sig = '***' if abs(row['Alpha t-stat']) > 2.58 else '**' if abs(row['Alpha t-stat']) > 1.96 else '*' if abs(row['Alpha t-stat']) > 1.64 else ''
    print(f"│      {symbol}: Alpha = {row['Alpha (Annual)']*100:>+5.2f}% {sig:3s} (t={row['Alpha t-stat']:>5.2f})                        │")

print(f"""
├──────────────────────────────────────────────────────────────────────────────┤
│  KEY FINDINGS                                                                │
│                                                                              │""")

best_sharpe = ranked.index[0]
best_calmar = metrics_df.sort_values('Calmar Ratio', ascending=False).index[0]
lowest_dd = metrics_df.sort_values('Max Drawdown', ascending=False).index[0]

print(f"│    • Best Risk-Adjusted (Sharpe): {best_sharpe}                                       │")
print(f"│    • Best Calmar Ratio: {best_calmar}                                                │")
print(f"│    • Lowest Max Drawdown: {lowest_dd}                                               │")

print(f"""
└──────────────────────────────────────────────────────────────────────────────┘

DETAILED METRICS:
""")

for symbol in ['SPY', 'QQQ', 'USMV']:
    m = metrics_df.loc[symbol]
    print(f"  {symbol}:")
    print(f"    Return: CAGR={m['CAGR']*100:.1f}%, Total={m['Total Return']*100:.1f}%")
    print(f"    Risk: Vol={m['Volatility']*100:.1f}%, MaxDD={m['Max Drawdown']*100:.1f}%")
    print(f"    Ratios: Sharpe={m['Sharpe Ratio']:.2f}, Sortino={m['Sortino Ratio']:.2f}, Calmar={m['Calmar Ratio']:.2f}")
    if symbol != 'SPY':
        print(f"    Benchmark: Beta={m['Beta']:.2f}, Alpha={m['Alpha']*100:.2f}%, IR={m['Information Ratio']:.2f}")
    print()

print("\n" + "="*80)
print("END OF PERFORMANCE ANALYSIS")
print("="*80)


---

## Summary & Key Takeaways

This notebook demonstrated **professional risk-adjusted performance analysis**:

### Performance Framework
1. **Return Metrics** - CAGR, total return, arithmetic mean
2. **Risk Metrics** - Volatility, drawdown, VaR, CVaR
3. **Risk-Adjusted** - Sharpe, Sortino, Calmar, Information Ratio
4. **Benchmark** - Alpha, Beta, tracking error, capture ratios

### Domain APIs Used
- `qj.eod.get_historical_prices`
- `local pandas performance analytics`

### Professional Best Practices
- Use multiple risk-adjusted metrics
- Analyze rolling performance for consistency
- Compare vs appropriate benchmarks
- Assess statistical significance of alpha
- Consider tail risk (VaR, CVaR, drawdown)